## **Students Contribution Table**


| Student ID   | Name                  | Percentage contribution |
|:-------------|:----------------------|:------------------------|
| 2024AA05399  | Santosh Kumar Prasad  | 100%                    |
| 2024AA05777  | Tajne Tushar Balasaheb| 100%                    |
| 2024AA05396  | Palanivel Subramanian | 100%                    |
| 2023ac05944  | AYUSH SINGH VERMA     | 100%                    |
| 2024AA05398  | Gajendra Sahani       | 100%                    |
```

# Face Modification and Generation using Generative Models



This assignment explores and compares multiple deep generative modeling
approaches for face image reconstruction, modification, and synthesis
using the CelebA dataset.

The objective is to understand how different latent representations
(continuous vs. discrete) influence reconstruction quality, attribute
manipulation, disentanglement, and sample realism.

The following models are implemented and analyzed:
- Variational Autoencoder (VAE)
- β-Variational Autoencoder (β-VAE)
- Vector Quantized VAE (VQ-VAE) with PixelCNN prior
- Generative Adversarial Network (GAN)

Through qualitative visualizations and comparative analysis, this
assignment highlights the trade-offs between reconstruction accuracy,
latent space interpretability, and realism in generative face modeling.


# Importing required python libraries

In [ ]:

# Import Statements for a PyTorch-based Deep Learning Project

# ======================
# Core Python Libraries
# ======================

import os
import math
import random
from pathlib import Path

# =========================
# Numerical & Data Handling
# =========================

import numpy as np
import pandas as pd

# ==============
# Visualization
# ==============

import matplotlib.pyplot as plt
from matplotlib import gridspec

# =============
# PyTorch Core
# =============

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ==================
# PyTorch Utilities
# ==================

from torch.utils.data import DataLoader, Dataset
from torch.nn.utils import clip_grad_norm_

# ===================================
# Torchvision (Datasets & Transforms)
# ===================================

import torchvision
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image

# ====================
# Progress Monitoring
# ====================

from tqdm import tqdm

# ================
# Reproducibility
# ================

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# =====================
# Device Configuration
# =====================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# Dataset: CelebA (Faces)



In this section, we load and preprocess the CelebA dataset, which consists of
celebrity face images along with attribute annotations.

### Preprocessing steps:
- Resize images to 64×64
- Convert images to tensors
- Normalize pixel values to [0, 1]

This dataset and preprocessing pipeline will be reused for all models:
VAE, β-VAE, VQ-VAE, and GAN.


In [ ]:
# ======================
# Image Transformations
# ======================
# Resize images to 64x64 and normalize to [0, 1]

transform = transforms.Compose([
    transforms.Resize((64, 64)),  # => Resize images to 64×64
    transforms.ToTensor()         # => Normalize pixel values to [0,1]
])

# ====================
# Load CelebA Dataset
# ====================
celeba_dataset = datasets.CelebA(
    root="./data",
    split="train",
    target_type="attr",
    transform=transform,
    download=True
)

# ======================
# Attribute Selection
# ======================
# CelebA attribute names (fixed order provided by the dataset)
celeba_attributes = celeba_dataset.attr_names

# Indices of commonly used attributes
smiling_idx = celeba_attributes.index("Smiling")
male_idx = celeba_attributes.index("Male")
eyeglasses_idx = celeba_attributes.index("Eyeglasses")

print("Selected attribute indices:")
print("Smiling:", smiling_idx)
print("Male:", male_idx)
print("Eyeglasses:", eyeglasses_idx)

# NOTE:
# Due to computational constraints, all models were trained on a subset of the
# CelebA dataset with a reduced number of epochs.

# ================================
# Use Subset of Dataset (Speed-up)
# ================================
from torch.utils.data import Subset

subset_size = 20000  # Using a subset of 20,000 images for faster training
indices = list(range(subset_size))
celeba_subset = Subset(celeba_dataset, indices)

# ============
# DataLoader
# ============
batch_size = 512  # Increased batch size for faster training

celeba_loader = DataLoader(
    celeba_subset,        # USING SUBSET HERE
    batch_size=batch_size,
    shuffle=True,
    num_workers=0, # speed up the data loading
    pin_memory=True
)

print(f"Number of training images (subset): {len(celeba_subset)}")



### Visual Inspection of Training Images

Before training any model, it is important to visually inspect a batch
of images to ensure that preprocessing is correct.


In [ ]:
# ============================
# Visualize a Batch of Images
# ============================

images, attributes = next(iter(celeba_loader))

# Take first 16 images for visualization
images = images[:16]

# Create a grid of images
image_grid = make_grid(images, nrow=4)

# Plot the images
plt.figure(figsize=(6, 6))
plt.imshow(image_grid.permute(1, 2, 0))  # Convert CHW to HWC
plt.axis("off")
plt.title("Sample CelebA Images (64x64)")
plt.show()


# Part A: Variational Autoencoder (VAE)



A Variational Autoencoder (VAE) is a generative model that learns a continuous,
structured latent representation of input data.

### Key components:
- Encoder: maps input image to latent distribution (mean and variance)
- Reparameterization trick: enables backpropagation through sampling
- Decoder: reconstructs image from latent sample

The VAE is trained using:
- Reconstruction loss (how close output is to input)
- KL divergence (regularizes latent space)


In [ ]:
# ================
# Encoder Network
# ================

class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        # Convolutional layers to extract spatial features
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),   # (B, 3, 64, 64) -> (B, 32, 32, 32)
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),  # -> (B, 64, 16, 16)
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), # -> (B, 128, 8, 8)
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1),# -> (B, 256, 4, 4)
            nn.ReLU()
        )

        # Fully connected layers for mean and log-variance
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)  # Flatten
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar


### Reparameterization Trick

To allow gradients to flow through random sampling, we sample latent variables
using the reparameterization trick:

z = μ + σ ⊙ ε, where ε ~ N(0, I)


In [ ]:
# =========================
# Reparameterization Trick
# =========================

def reparameterize(mu, logvar):
    # Standard deviation
    std = torch.exp(0.5 * logvar)

    # Random noise
    eps = torch.randn_like(std)

    # Sampled latent vector
    z = mu + eps * std
    return z


In [ ]:
# ================
# Decoder Network
# ================

class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        # Project latent vector to feature map
        self.fc = nn.Linear(latent_dim, 256 * 4 * 4)

        # Transposed convolutions to reconstruct image
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # -> (B, 128, 8, 8)
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # -> (B, 64, 16, 16)
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),    # -> (B, 32, 32, 32)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),     # -> (B, 3, 64, 64)
            nn.Sigmoid()  # Output in [0, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 256, 4, 4)
        x = self.deconv(x)
        return x


### VAE Model

The full VAE combines the encoder, reparameterization, and decoder
into a single forward pass.


In [ ]:
# ===========
# VAE Model
# ===========

class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        # Encode input image
        mu, logvar = self.encoder(x)

        # Sample latent vector
        z = reparameterize(mu, logvar)

        # Reconstruct image
        recon_x = self.decoder(z)

        return recon_x, mu, logvar


### VAE Loss Function

The VAE loss consists of:
- Reconstruction loss (Binary Cross-Entropy)
- KL divergence loss


In [ ]:
# ==================
# VAE Loss Function
# ==================

def vae_loss(recon_x, x, mu, logvar):
    # Reconstruction loss
    recon_loss = F.binary_cross_entropy(
        recon_x, x, reduction="sum"
    )

    # KL divergence
    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    return recon_loss + kl_loss


### Training Loop for VAE

This loop trains the VAE using minibatch gradient descent.


In [ ]:
# ==================
# VAE Training Loop
# ==================

def train_vae(model, dataloader, optimizer, epochs):
    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for x, _ in tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}"):
            x = x.to(device)

            optimizer.zero_grad()

            recon_x, mu, logvar = model(x)
            loss = vae_loss(recon_x, x, mu, logvar)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader.dataset)
        print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")


### Initialize and Train the VAE


In [ ]:
# =====================
# Model Initialization
# =====================

latent_dim = 128
vae = VAE(latent_dim).to(device)

optimizer = optim.Adam(vae.parameters(), lr=1e-3)

# Example training call (run when ready)
train_vae(vae, celeba_loader, optimizer, epochs=10) # ajust epochs as needed.


### VAE Reconstruction Visualization

This section visualizes original images alongside their reconstructions
produced by the trained VAE.


In [ ]:
# ==============================
# Visualize VAE Reconstructions
# ==============================
def visualize_reconstructions(model, dataloader, num_images=8):
    model.eval()

    with torch.no_grad():
        images, _ = next(iter(dataloader))
        images = images[:num_images].to(device)

        recon_images, _, _ = model(images)

        comparison = torch.cat([images, recon_images], dim=0)
        grid = make_grid(comparison, nrow=num_images)

        plt.figure(figsize=(2 * num_images, 4))
        plt.imshow(grid.permute(1, 2, 0).cpu())
        plt.axis("off")
        plt.title("Top: Original | Bottom: Reconstruction")
        plt.show()


visualize_reconstructions(vae, celeba_loader)


### Latent Space Interpolation

Latent interpolation demonstrates the smoothness of the learned latent
space by interpolating between two encoded face images.


In [ ]:
# =========================
# Latent Interpolation
# =========================
def latent_interpolation(model, dataloader, steps=10):
    model.eval()

    with torch.no_grad():
        images, _ = next(iter(dataloader))
        img1 = images[0:1].to(device)
        img2 = images[1:2].to(device)

        mu1, _ = model.encoder(img1)
        mu2, _ = model.encoder(img2)

        interpolations = []
        for alpha in torch.linspace(0, 1, steps):
            z = (1 - alpha) * mu1 + alpha * mu2
            interpolations.append(model.decoder(z))

        interpolations = torch.cat(interpolations, dim=0)
        grid = make_grid(interpolations, nrow=steps)

        plt.figure(figsize=(2 * steps, 2))
        plt.imshow(grid.permute(1, 2, 0).cpu())
        plt.axis("off")
        plt.title("Latent Interpolation")
        plt.show()


latent_interpolation(vae, celeba_loader)


### Attribute Modification via Latent Vector Arithmetic

Semantic attributes such as smiling are manipulated by computing
difference vectors in latent space and applying them to new samples.


In [ ]:
# ==================================
# Compute Attribute Direction Vector
# ==================================
def compute_attribute_vector(model, dataloader, attr_index, num_samples=500):
    model.eval()

    latent_pos = []
    latent_neg = []

    with torch.no_grad():
        for images, attrs in dataloader:
            images = images.to(device)
            attrs = attrs[:, attr_index]

            mu, _ = model.encoder(images)

            for i in range(len(attrs)):
                if attrs[i] == 1 and len(latent_pos) < num_samples:
                    latent_pos.append(mu[i])
                elif attrs[i] == 0 and len(latent_neg) < num_samples:
                    latent_neg.append(mu[i])

            if len(latent_pos) >= num_samples and len(latent_neg) >= num_samples:
                break

    latent_pos = torch.stack(latent_pos)
    latent_neg = torch.stack(latent_neg)

    return latent_pos.mean(dim=0) - latent_neg.mean(dim=0)


In [ ]:
# ===============================
# Apply Attribute Modification
# ===============================
def apply_attribute(model, dataloader, attr_vector, scale=1.5):
    model.eval()

    with torch.no_grad():
        images, _ = next(iter(dataloader))
        image = images[0:1].to(device)

        mu, _ = model.encoder(image)

        modified_z = mu + scale * attr_vector.unsqueeze(0)

        original = model.decoder(mu)
        modified = model.decoder(modified_z)

        comparison = torch.cat([image, original, modified], dim=0)
        grid = make_grid(comparison, nrow=3)

        plt.figure(figsize=(6, 2))
        plt.imshow(grid.permute(1, 2, 0).cpu())
        plt.axis("off")
        plt.title("Original | Reconstruction | Attribute Modified")
        plt.show()


smile_vector = compute_attribute_vector(vae, celeba_loader, smiling_idx)
apply_attribute(vae, celeba_loader, smile_vector)

# Part B: β-VAE

In [ ]:
# =========================
# β-VAE Loss Function
# =========================

# The β-VAE loss modifies the standard VAE objective by scaling the KL divergence term using a hyperparameter β.

def beta_vae_loss(recon_x, x, mu, logvar, beta):
    # Reconstruction loss
    recon_loss = F.binary_cross_entropy(
        recon_x, x, reduction="sum"
    )

    # KL divergence
    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    return recon_loss + beta * kl_loss


In [ ]:
# =========================
# β-VAE Training Loop
# =========================

# Each β-VAE model is trained independently using a different value of β.

def train_beta_vae(model, dataloader, optimizer, beta, epochs):
    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for x, _ in tqdm(dataloader, desc=f"β={beta} | Epoch {epoch+1}/{epochs}"):
            x = x.to(device)

            optimizer.zero_grad()
            recon_x, mu, logvar = model(x)
            loss = beta_vae_loss(recon_x, x, mu, logvar, beta)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader.dataset)
        print(f"β={beta} | Epoch {epoch+1}, Loss: {avg_loss:.4f}")


## Train models with different β values

β-VAE is a variant of the standard VAE that introduces a hyperparameter β
to control the strength of the KL-divergence term.

Increasing β encourages disentangled latent representations, often at the
cost of reconstruction quality.

In this section, β-VAE models are trained and analyzed for β ∈ {2, 4, 10}.


In [ ]:
# =========================
# Train β-VAE Models
# =========================
beta_values = [2, 4, 10]
beta_vae_models = {}

for beta in beta_values:
    model = VAE(latent_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    train_beta_vae(model, celeba_loader, optimizer, beta, epochs=5) # adjust epochs as needed.

    beta_vae_models[beta] = model


### Latent Traversal for β-VAE

Latent traversal is performed by varying one latent dimension at a time
while keeping all other dimensions fixed. This helps visualize the
semantic meaning captured by individual latent variables.


In [ ]:
# =========================
# Latent Traversal
# =========================
def latent_traversal(model, dim, beta, steps=7, value_range=(-3, 3)):
    model.eval()

    with torch.no_grad():
        z = torch.zeros(1, latent_dim).to(device)
        values = torch.linspace(value_range[0], value_range[1], steps)

        images = []
        for val in values:
            z[0, dim] = val
            img = model.decoder(z)
            images.append(img)

        images = torch.cat(images, dim=0)
        grid = make_grid(images, nrow=steps)

        plt.figure(figsize=(2 * steps, 2))
        plt.imshow(grid.permute(1, 2, 0).cpu())
        plt.axis("off")
        plt.title(f"Latent Traversal (β={beta}, Dimension={dim})")
        plt.show()


In [ ]:
# Latent traversal for different β values
latent_traversal(beta_vae_models[2], dim=10, beta=2)
latent_traversal(beta_vae_models[4], dim=10, beta=4)
latent_traversal(beta_vae_models[10], dim=10, beta=10)



### Identification of Attribute-Controlling Latent Dimensions

In this task, latent traversal is used to identify which latent dimensions
control semantic facial attributes such as smiling, gender, and pose.

By varying one latent dimension at a time and visually inspecting the
generated images, dimensions that correlate with specific attributes
can be identified qualitatively.


# Note: Beta value and it's behaviour

| β value    | Typical behavior                              |
| ---------- | --------------------------------------------- |
| **β = 2**  | Good reconstruction, weak disentanglement     |
| **β = 4**  | **Balanced** reconstruction + disentanglement |
| **β = 10** | Strong disentanglement, poor reconstruction   |


In [ ]:
# ==========================================
# Identify Attribute-Controlling Dimensions
# ==========================================

# Using β = 4 as a representative model

beta = 4
model = beta_vae_models[beta]

# Inspect multiple latent dimensions
latent_traversal(model, dim=5, beta=beta)    # Possible expression change
latent_traversal(model, dim=12, beta=beta)   # Possible gender-related features
latent_traversal(model, dim=25, beta=beta)   # Possible pose or orientation


### Reconstruction vs Disentanglement Trade-off

This task analyzes the trade-off between reconstruction quality and latent
space disentanglement in β-VAE models.

By increasing the value of β, stronger regularization is imposed on the
latent space, which encourages disentanglement but can degrade the quality
of image reconstructions.


### Observations

- For lower β values (e.g., β = 2), reconstructions closely resemble the
  original images, but latent dimensions tend to encode multiple factors
  of variation, resulting in weaker disentanglement.

- As β increases (e.g., β = 4 and β = 10), reconstructions lose fine-grained
  details, but individual latent dimensions become more interpretable and
  are more strongly associated with specific semantic attributes.

- This behavior highlights a clear trade-off in β-VAE models: improving
  disentanglement comes at the cost of reconstruction fidelity.


# Part C: VQ-VAE with PixelCNN Prior

# Part D: GAN for Face Generation

# Part E: Comparative Analysis